In [ ]:
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np
sys.path.append("/Users/gabriel/Desktop/Quant_Project/Quant-Summer-Project")
from scipy.stats import norm
from tests.fixtures.synthetic_data import synthetic_actual_df, synthetic_previous_df
from data import fetcher
from data.fetcher import (
fetch_data,
get_tommorows_wheather,
pair_dataframes,compute_forecast_error,
fetch_previous_forecast_data,
get_daily_max,
)

from models.bayesian_model import (
bayesian_interference,
posterior_probability,
)

from models.baseline import (
gaussian_probability
)
from models.kde_model import (
kde_estimate
)
from data.cleaner import clean_data
from data.loader import (
add_event_column,
filter_summer
)
from config.settings import (
HISTORICAL_START,
HISTORICAL_END,
DEFAULT_CITY,
TOMMORROWS_DATE,
FORECAST_START,
FORECAST_END 
)

fetcher.fetch_data = lambda *args, **kwargs: synthetic_actual_df()
fetcher.fetch_previous_forecast_data = lambda *args, **kwargs: synthetic_previous_df()
fetcher.get_tommorows_wheather = lambda *args, **kwargs: 30


df_raw = fetch_data(HISTORICAL_START,HISTORICAL_END)


df_clean = clean_data(df_raw)


df_event = add_event_column(df_clean)


df_temp_summer = filter_summer(df_event)

df_temp_list = df_temp_summer["temperature_2m_max"].to_list()

data = []
sigma_posterior, _, _, my_posterior, _, _ = bayesian_interference(df_temp_list)
for i in range(25,36):
    bucket_center = i +0.5
    P_baseline,_,_ = gaussian_probability(df_temp_list,i,i+1)
    P_KDE = kde_estimate(df_temp_list,i,i+1)
    #To not run out of API calls
    P_Bayesian_Inference = norm.cdf((i+1 - my_posterior)/sigma_posterior) - norm.cdf((i - my_posterior)/sigma_posterior)
    data.append( (bucket_center, "Gauss", P_baseline))
    data.append( (bucket_center, "KDE", P_KDE))
    data.append( (bucket_center, "Bayesian",P_Bayesian_Inference ))

long_df = pl.DataFrame(data,schema = ['Bucket Center', 'Model', 'Probability'],orient = "row" )


sns.barplot(data=long_df, x="Bucket Center", y="Probability", hue="Model")




This chart shows where the three models disagree, but it doesn't tell us which one is right — there's no ground truth on it, just three sets of assumptions compared to each other.
For Gauss vs KDE specifically, we already found the answer in notebook 03: comparing both against the actual empirical histogram of real summer days showed KDE tracks reality better at the "shoulder" buckets, while Gaussian does better right at the center and out in the extreme tail — a result of the underlying data being a mixture of three differently-shaped months (June/July/August) that a single symmetric Gaussian can't fully capture.
For Bayesian, this kind of check doesn't even apply the same way — it isn't predicting "the general shape of summer," it's making a claim about one specific day. The only real way to judge whether it's any good is to check it across many days: of all the times it said "70% chance," did the event actually happen about 70% of the time? That's a calibration question, not a distribution-shape question, and it's what the evaluation notebook (Brier score, log loss, calibration curves) is for.
Here we can see what two separate questions the models answer. KDE and Gauss is general climatological and Bayesian is  one particular day's . So Gauss and KDE represent what typical for any summer day in general. The bayesian bars represent specific days forecast updated estimate , whatever days forecast happend to be plugged in.
One thing to be aware of if you ran this tommorow then Gauss and KDE would look the same thus they only depend on historical data where as Bayesian would not, thus it depens on tommorows forecast.
